# Parameter-Efficient Fine-Tuning of LLaVA for Chest Radiology Report Generation

**Dataset:** MIMIC-CXR (simhadrisadaram/mimic-cxr-dataset, study-level pairs)  
**Model:** LLaVA-1.5-7B · **Quantisation:** 4-bit NF4 · **Adapters:** QLoRA rank-16

---

### Prerequisites

Run `preprocess_local.py` on your local machine before this notebook.  
It produces a single zip file containing:
- `images/` — 336×336 JPEG chest X-rays, one per study
- `train.json` — list of `{subject_id, study_id, image, report}` dicts
- `val.json`   — same format, held-out split

Upload that zip to the **root** of your Google Drive, then run this notebook from top to bottom.

---

### Optimisation Stack

| Technique | Purpose |
|-----------|--------|
| 4-bit NF4 + double quantisation | Model footprint ~14 GB → ~4 GB |
| QLoRA rank-16 | Only ~0.5 % of parameters trained |
| Paged AdamW 8-bit | Optimizer states page to CPU; prevents OOM spikes |
| Gradient checkpointing | Recomputes activations on backward pass; saves peak activation memory |
| NEFTune noise (α=5) | Embedding regularisation; improves generation quality |
| Group-by-length batching | Minimises per-batch padding tokens |
| Cosine LR with 3 % warmup | Stable convergence from cold start |
| Frozen vision tower | CLIP ViT already generalises well; updating it wastes capacity |

---
## Step 1 — Global Configuration

All hyperparameters are centralised here. Adjust paths and training knobs in one place before running any other cell.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
ZIP_PATH       = "/content/drive/MyDrive/preprocessed_cxr.zip"   # adjust if needed
DATA_DIR       = "/content/cxr_data"
CHECKPOINT_DIR = "/content/drive/MyDrive/llava_cxr_checkpoints"
LORA_SAVE_DIR  = "/content/drive/MyDrive/llava_cxr_lora_final"

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_ID       = "llava-hf/llava-1.5-7b-hf"

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_R         = 16     # rank; higher = more capacity, more memory
LORA_ALPHA     = 32     # scaling = alpha / r; 2× rank is a stable default
LORA_DROPOUT   = 0.05

# ── Training ───────────────────────────────────────────────────────────────
MAX_SEQ_LENGTH      = 512   # covers > 95 % of report lengths in this dataset
PER_DEVICE_BATCH    = 2     # physical micro-batch per GPU
GRAD_ACCUM_STEPS    = 8     # effective batch size = 2 × 8 = 16
LEARNING_RATE       = 2e-4
WARMUP_RATIO        = 0.03
MAX_TRAIN_SAMPLES   = None  # set e.g. 500 for a quick smoke-test

print("Configuration loaded.")

---
## Step 2 — Install Dependencies

Pins to the latest stable releases. `bitsandbytes` provides the CUDA kernels for 4-bit quantisation and the 8-bit paged optimizer. `peft` handles LoRA injection. `accelerate` manages device mapping and mixed-precision transparently.

In [ ]:
!pip install -q --upgrade transformers accelerate peft bitsandbytes pillow tqdm

---
## Step 3 — Mount Drive and Extract Data

The archive is extracted once into `/content/cxr_data`. The check at the top makes this cell idempotent — safe to re-run after a session interruption without wasting time re-extracting.

In [ ]:
import os, json, zipfile, torch
from google.colab import drive

drive.mount("/content/drive")

if not os.path.exists(f"{DATA_DIR}/train.json"):
    print("Extracting archive...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATA_DIR)
    print("Done.")
else:
    print("Data already extracted — skipping.")

train_data = json.load(open(f"{DATA_DIR}/train.json"))
val_data   = json.load(open(f"{DATA_DIR}/val.json"))

print(f"Training samples : {len(train_data):,}")
print(f"Validation samples: {len(val_data):,}")
print(f"GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND — check runtime'}")

# Inspect one sample to verify preprocessing output is correct
s = train_data[0]
print(f"\nSample record:")
print(f"  subject_id : {s['subject_id']}")
print(f"  study_id   : {s['study_id']}")
print(f"  image path : {s['image']}")
print(f"  report[:80]: {s['report'][:80]}")

---
## Step 4 — Load LLaVA-1.5-7B with 4-bit Quantisation

**NF4 (NormalFloat-4)** is an information-theoretically optimal 4-bit data type for weights that follow a normal distribution — which pre-trained transformer weights do. It minimises rounding error for the same bit width compared to uniform INT4.

**Double quantisation** compresses the quantisation scaling constants themselves into 8-bit, saving ~0.4 bits per parameter (≈ 0.4 GB for a 7B model) at negligible accuracy cost.

**`low_cpu_mem_usage=True`** constructs each layer one at a time rather than allocating the full model in host RAM before moving it, preventing out-of-memory on machines with limited RAM.

TF32 is enabled for Ampere+ GPUs (A100, RTX 30/40-series, A10). It gives ~8× throughput over FP32 for matrix multiplications with almost no numerical difference.

In [ ]:
import torch
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# TF32 is a no-op on GPUs that don't support it (e.g. T4), safe to leave on
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.float16,
    bnb_4bit_use_double_quant = True,
)

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map          = "auto",
    torch_dtype         = torch.float16,
    low_cpu_mem_usage   = True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print(f"Parameters : {sum(p.numel() for p in model.parameters())/1e9:.2f} B")
print(f"VRAM used  : {torch.cuda.memory_allocated()/1e9:.2f} GB")

---
## Step 5 — Inject QLoRA Adapters

LoRA decomposes each weight update into two low-rank matrices: `W += BA` where `B ∈ R^{d×r}` and `A ∈ R^{r×k}`. With `r=16`, this represents each projection update with 16 × (d+k) numbers instead of d×k — roughly 50× fewer parameters per layer.

All seven projection matrices in each transformer block are targeted: the four attention projections (`q, k, v, o`) and the three MLP projections (`gate, up, down` in the SwiGLU activation). Covering all projections gives the model full flexibility to reroute representations without the vision tower's gradient flowing at all.

The vision tower (CLIP ViT-L/14) is explicitly frozen after adapter injection. CLIP was pre-trained on 400M image-text pairs and already produces strong visual tokens — updating it on a single medical domain risks forgetting cross-domain visual priors without measurable report quality gain.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    task_type      = "CAUSAL_LM",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)

# Freeze vision tower — LoRA may have matched projections inside it as well
for name, param in model.named_parameters():
    if "vision_tower" in name:
        param.requires_grad = False

model.print_trainable_parameters()
print(f"VRAM after adapter injection: {torch.cuda.memory_allocated()/1e9:.2f} GB")

---
## Step 6 — Dataset Construction

The training objective is **causal language modelling conditioned on an image**: given the visual tokens and the instruction prompt, the model should produce the radiology report token-by-token.

**Prompt masking** (`labels[: prompt_len] = -100`): PyTorch's cross-entropy ignores positions labelled `-100`. This means the model is penalised only for the report it generates, never for repeating the fixed instruction. Without this masking the model would waste capacity memorising the prompt string instead of learning clinical vocabulary.

**Conversation format**: LLaVA-1.5 was instruction-tuned using `USER:` / `ASSISTANT:` separators. Staying within this exact template reuses the pre-trained instruction-following conditioning and converges faster than using a custom prompt format.

**`StackCollator`**: Because the dataset already pads to `MAX_SEQ_LENGTH` in `__getitem__`, the collator only needs to stack tensors — no secondary padding pass is needed, keeping data loading overhead minimal.

In [ ]:
import torch
from PIL import Image
from torch.utils.data import Dataset

PROMPT = (
    "USER: <image>\n"
    "Analyze this chest X-ray and write a detailed radiology report "
    "including findings and impression.\n"
    "ASSISTANT: "
)

class ChestXrayDataset(Dataset):
    def __init__(self, records, proc, data_dir, max_len):
        self.records    = records
        self.proc       = proc
        self.data_dir   = data_dir
        self.max_len    = max_len
        # Pre-compute prompt token count once — used for label masking every item
        self.prompt_len = len(proc.tokenizer(PROMPT, add_special_tokens=True)["input_ids"])

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec   = self.records[idx]
        image = Image.open(os.path.join(self.data_dir, rec["image"])).convert("RGB")
        text  = PROMPT + rec["report"].strip() + self.proc.tokenizer.eos_token

        enc = self.proc(
            text           = text,
            images         = image,
            return_tensors = "pt",
            padding        = "max_length",
            max_length     = self.max_len,
            truncation     = True,
        )

        ids    = enc["input_ids"].squeeze(0)
        labels = ids.clone()
        labels[: self.prompt_len] = -100               # mask instruction tokens
        pad_id = self.proc.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100            # mask padding tokens

        return {
            "input_ids":      ids,
            "attention_mask": enc["attention_mask"].squeeze(0),
            "pixel_values":   enc["pixel_values"].squeeze(0),
            "labels":         labels,
        }


class StackCollator:
    """Collates pre-padded batches by stacking — no extra padding pass."""
    def __call__(self, features):
        return {k: torch.stack([f[k] for f in features]) for k in features[0]}


train_recs    = train_data[:MAX_TRAIN_SAMPLES] if MAX_TRAIN_SAMPLES else train_data
train_dataset = ChestXrayDataset(train_recs, processor, DATA_DIR, MAX_SEQ_LENGTH)
val_dataset   = ChestXrayDataset(val_data,   processor, DATA_DIR, MAX_SEQ_LENGTH)

print(f"Train: {len(train_dataset):,}  |  Val: {len(val_dataset):,}")
s = train_dataset[0]
print(f"  input_ids    : {s['input_ids'].shape}")
print(f"  pixel_values : {s['pixel_values'].shape}")
print(f"  active labels: {(s['labels'] != -100).sum().item()} tokens")

---
## Step 7 — Training

The training loop uses several co-operating techniques to maximise gradient quality within a fixed memory budget:

**`paged_adamw_8bit`**: The standard AdamW optimizer stores two momentum tensors per trainable parameter in FP32. For a 7B model, even 0.5% trainable parameters means ~35M params × 8 bytes = ~280 MB of optimizer state. The paged 8-bit variant quantises these states and pages unused blocks to CPU RAM, avoiding the hard OOM that occurs when optimizer states spike GPU memory during the first update step.

**`neftune_noise_alpha=5`**: During the embedding lookup, small Gaussian noise is added to the embedding vectors. This acts as a data augmentation in embedding space and has been empirically shown to improve instruction-following and text quality without any hyperparameter search cost.

**`group_by_length=True`**: Batches are assembled from samples with similar sequence lengths. Because the dataset has high length variance (short normal-finding reports vs. long multi-study comparison reports), this significantly reduces the fraction of each batch consumed by padding tokens.

**Auto-resume**: The training cell scans the checkpoint directory for the latest saved checkpoint and passes it to `trainer.train()`. This makes the cell fully resumable after any session interruption — just re-run all preceding cells, then re-run this one.

In [ ]:
from transformers import TrainingArguments, Trainer

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir                    = CHECKPOINT_DIR,
    num_train_epochs              = 1,

    # Batch size and accumulation
    per_device_train_batch_size   = PER_DEVICE_BATCH,
    per_device_eval_batch_size    = PER_DEVICE_BATCH,
    gradient_accumulation_steps   = GRAD_ACCUM_STEPS,

    # Optimizer and schedule
    optim                         = "paged_adamw_8bit",   # 4× less optimizer memory
    learning_rate                 = LEARNING_RATE,
    weight_decay                  = 0.01,
    warmup_ratio                  = WARMUP_RATIO,
    lr_scheduler_type             = "cosine",
    max_grad_norm                 = 1.0,

    # Memory and precision
    fp16                          = True,
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},

    # Quality improvements
    neftune_noise_alpha           = 5,       # embedding-space regularisation
    group_by_length               = True,    # reduces padding waste per batch

    # Logging and checkpointing
    logging_steps                 = 25,
    eval_strategy                 = "steps",
    eval_steps                    = 1000,
    save_steps                    = 500,     # saves to Drive for resume support
    save_total_limit              = 3,       # keep only latest 3 checkpoints
    save_safetensors              = True,

    # Data loading
    dataloader_num_workers        = 2,
    dataloader_pin_memory         = True,
    dataloader_persistent_workers = True,

    remove_unused_columns         = False,
    report_to                     = "none",
    seed                          = 42,
)

# Auto-detect latest checkpoint for seamless resume
ckpts = sorted(
    [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("checkpoint-")],
    key=lambda x: int(x.split("-")[1])
)
resume_ckpt = os.path.join(CHECKPOINT_DIR, ckpts[-1]) if ckpts else None
print("Resuming from:", resume_ckpt or "scratch")

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    data_collator = StackCollator(),
)

trainer.train(resume_from_checkpoint=resume_ckpt)
print("Training complete.")

---
## Step 8 — Save LoRA Adapters

Only the adapter matrices and the processor config are saved — not the 7B base model. This results in a ~50-100 MB artifact rather than ~14 GB. Reproducibility is preserved: the base model is always re-fetched from the HuggingFace hub, and the adapter delta carries all learned domain-specific knowledge.

In [ ]:
os.makedirs(LORA_SAVE_DIR, exist_ok=True)
model.save_pretrained(LORA_SAVE_DIR)
processor.save_pretrained(LORA_SAVE_DIR)

size_mb = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fns in os.walk(LORA_SAVE_DIR) for f in fns
) / 1e6
print(f"Saved to : {LORA_SAVE_DIR}")
print(f"Size     : {size_mb:.1f} MB")

---
## Step 9 — Qualitative Inference

Greedy decoding (`do_sample=False`) is used here. It is deterministic and produces the highest-probability token at each step — the right choice when the goal is to verify that the model has learned correct clinical structure (Findings / Impression) and domain vocabulary, rather than diversity.

For quantitative evaluation, compute BLEU-4, ROUGE-L, and CheXbert F1 over the full validation set.

In [ ]:
from PIL import Image

model.eval()

INFER_PROMPT = (
    "USER: <image>\n"
    "Analyze this chest X-ray and write a detailed radiology report "
    "including findings and impression.\n"
    "ASSISTANT: "
)

for i, rec in enumerate(val_data[:3]):
    img = Image.open(os.path.join(DATA_DIR, rec["image"])).convert("RGB")
    inp = processor(text=INFER_PROMPT, images=img, return_tensors="pt")
    inp = {k: v.to(model.device) for k, v in inp.items()}

    with torch.no_grad():
        out_ids = model.generate(**inp, max_new_tokens=300, do_sample=False)

    generated = processor.tokenizer.decode(out_ids[0], skip_special_tokens=True)
    response  = generated.split("ASSISTANT:")[-1].strip()

    print(f"\n{'─'*60}  Sample {i+1}  (subject={rec['subject_id']} study={rec['study_id']})")
    print(f"GENERATED:\n{response}")
    print(f"\nGROUND TRUTH:\n{rec['report'][:400]}")

---
## Step 10 — Load Saved Adapters (New Session)

To restore the fine-tuned model in a future session without retraining:
1. Run Steps 1–5 (configuration, install, mount, base model load, QLoRA prep).
2. Run this cell to merge the saved adapter delta onto the base model.

In [ ]:
from peft import PeftModel

# Steps 1-5 (config, install, mount, model load, QLoRA) must be run first.
model = PeftModel.from_pretrained(model, LORA_SAVE_DIR)
model.eval()
print(f"Adapters loaded from : {LORA_SAVE_DIR}")
print("Proceed to Step 9 for inference.")